# Human-in-the-Loop Review Comparison for IDS

This Colab notebook compares human review trigger strategies for intrusion detection decisions.

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn

## 3) Load Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RNG_SEED = 42
np.random.seed(RNG_SEED)

data_path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
print('Files:', os.listdir(data_path))

X_test = np.load(os.path.join(data_path, 'X_test.npy'))
y_test = np.load(os.path.join(data_path, 'y_test.npy')).reshape(-1)

if X_test.ndim == 3 and X_test.shape[-1] == 1:
    X_test = X_test[..., 0]

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

## 4) Generate Attack Predictions

In [ ]:
def simulate_predictions(X, y, n_samples=500, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(y), size=min(n_samples, len(y)), replace=False)

    attack_map = {0: 'BENIGN', 1: 'DOS', 2: 'PORTSCAN', 3: 'BRUTEFORCE', 4: 'DDOS'}
    rows = []
    for sample_id, i in enumerate(idx):
        true_label = int(y[i])
        attack_type = attack_map.get(true_label, f'ATTACK_{true_label}')

        feature_signal = float(np.clip(np.mean(np.abs(X[i])), 0.0, 1.0))
        confidence = float(np.clip(0.65 * feature_signal + 0.25 * (true_label > 0) + rng.normal(0, 0.12), 0.0, 1.0))
        risk_score = float(np.clip(0.6 * confidence + 0.3 * (true_label > 0) + rng.normal(0, 0.1), 0.0, 1.0))

        # Proxy graph similarity: lower values can indicate novelty for this notebook setup
        graph_similarity = float(np.clip(rng.beta(2, 5) if true_label > 0 else rng.beta(5, 2), 0.0, 1.0))

        if risk_score > 0.8:
            decision = 'BLOCK_IP'
        elif risk_score > 0.6:
            decision = 'RATE_LIMIT'
        elif risk_score > 0.4:
            decision = 'ALERT_ADMIN'
        else:
            decision = 'ALLOW'

        rows.append({
            'sample_id': int(sample_id),
            'true_label': true_label,
            'attack_type': attack_type,
            'confidence': confidence,
            'risk_score': risk_score,
            'graph_similarity': graph_similarity,
            'decision': decision
        })

    return pd.DataFrame(rows)

pred_df = simulate_predictions(X_test, y_test, n_samples=500, seed=RNG_SEED)
pred_df.head()

## 5) Trigger Human Review Based on Strategies

In [ ]:
def confidence_trigger(row, threshold=0.65):
    return row['confidence'] < threshold

def uncertainty_trigger(row, low=0.45, high=0.65):
    return low <= row['risk_score'] <= high

def novelty_trigger(row, graph_sim_threshold=0.35):
    # Lower graph similarity indicates potential novelty
    return row['graph_similarity'] < graph_sim_threshold

def run_strategy(df, strategy_name):
    temp = df.copy()
    if strategy_name == 'Confidence Threshold Trigger':
        temp['review_required'] = temp.apply(confidence_trigger, axis=1)
    elif strategy_name == 'Risk Uncertainty Trigger':
        temp['review_required'] = temp.apply(uncertainty_trigger, axis=1)
    elif strategy_name == 'Novel Attack Trigger (Graph Similarity)':
        temp['review_required'] = temp.apply(novelty_trigger, axis=1)
    else:
        raise ValueError('Unknown strategy')
    return temp

strategies = [
    'Confidence Threshold Trigger',
    'Risk Uncertainty Trigger',
    'Novel Attack Trigger (Graph Similarity)'
]

## Human Input (yes/no) and Decision Storage

In [ ]:
# Ask user once for approval behavior in this demo run
user_input = input('Approve system decision? yes/no: ').strip().lower()
if user_input not in ['yes', 'no']:
    user_input = 'yes'

# Store human decisions only for reviewed samples
def apply_human_feedback(df, approval):
    out = df.copy()
    out['human_decision'] = np.where(out['review_required'], approval, 'auto_accept')
    return out

human_decision_store = {}
for s in strategies:
    reviewed = run_strategy(pred_df, s)
    reviewed = apply_human_feedback(reviewed, user_input)
    human_decision_store[s] = reviewed

print('Stored decisions for strategies:', list(human_decision_store.keys()))
human_decision_store[strategies[0]].head()

## Compare Review Frequency and False Positive Reduction

In [ ]:
def compute_metrics(df):
    review_frequency = float(df['review_required'].mean())

    # Baseline false positives: benign samples with mitigation actions
    benign = df[df['true_label'] == 0]
    baseline_fp = ((benign['decision'] != 'ALLOW')).sum()

    # Reviewed benign alerts assumed corrected if human says no
    if len(benign) == 0:
        fp_reduction = 0.0
    else:
        corrected_fp = ((benign['review_required']) & (benign['human_decision'] == 'no') & (benign['decision'] != 'ALLOW')).sum()
        fp_reduction = float(corrected_fp / max(int(baseline_fp), 1))

    return review_frequency, fp_reduction

rows = []
for s in strategies:
    d = human_decision_store[s]
    rf, fpr = compute_metrics(d)
    rows.append({'Method': s, 'ReviewFrequency': rf, 'FalsePositiveReduction': fpr})

metrics_df = pd.DataFrame(rows).sort_values(by=['FalsePositiveReduction', 'ReviewFrequency'], ascending=[False, True])
metrics_df

## Comparison Charts

In [ ]:
sns.set(style='whitegrid')

plt.figure(figsize=(10, 4))
sns.barplot(data=metrics_df, x='Method', y='ReviewFrequency', palette='Blues_d')
plt.ylim(0, 1)
plt.title('Review Frequency by Strategy')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.barplot(data=metrics_df, x='Method', y='FalsePositiveReduction', palette='Greens_d')
plt.ylim(0, 1)
plt.title('False Positive Reduction by Strategy')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
best = metrics_df.iloc[0]
print('Best strategy:', best['Method'])
print(best.to_string())